# Création des labels

# Imports

In [20]:
import os
import pandas as pd

In [21]:
attack_ips = {"192.168.1.8", "192.168.1.9", "192.168.1.10", "192.168.1.11"}

attack_dir_flows='csv/flows_attack'
benign_dir_flows='csv/flows_benign'
processed_attack='csv/processed/labelled_attack'
processed_benign='csv/processed/labelled_benign'
output='csv/processed/labelled_dataset.csv'

In [22]:
def drop_arp_packets(df, protocol_col='protocol'):
        """
        Supprime toutes les lignes où le protocole est ARP.
        - Gère la casse (arp / ARP / Arp)
        - Gère les valeurs manquantes
        - Gère les champs multiples (ex: 'ARP, LLDP' ou 'ICMP; ARP')
        """
        if protocol_col not in df.columns:
                return df

        protos = (
                df[protocol_col]
                .astype(str)
                .str.strip()
                .str.lower()
        )

        is_not_arp = ~protos.str.contains(r'(?:^|[,\s;/|])arp(?:$|[,\s;/|])', regex=True)

        return df[is_not_arp].copy()

In [23]:

def label_and_save(directory, out_dir, default_label):
        '''Labels CSV files based on IP addresses and saves them to the output directory.'''
        os.makedirs(out_dir, exist_ok=True)
        for f in os.listdir(directory):
                if f.endswith('.csv'):
                        df = pd.read_csv(os.path.join(directory, f))
                        df = drop_arp_packets(df)
                        df['label'] = df.apply(lambda r: 'MALICIOUS' if (str(r.get('src')).strip() in attack_ips or str(r.get('dst')).strip() in attack_ips) else default_label, axis=1)
                        f = f.replace('.csv', '_labelled.csv')
                        df.to_csv(os.path.join(out_dir, f), index=False)
                        print(f'Labeled: {f}')

In [24]:
label_and_save(attack_dir_flows, processed_attack, 'MALICIOUS')
label_and_save(benign_dir_flows, processed_benign, 'BENIGN')

Labeled: vertx_flows_merged_attack_2026-03-10_16-48-34_labelled.csv
Labeled: vertx_flows_merged_benign_2026-03-10_16-48-34_labelled.csv


# Merge des fichiers CSV

In [25]:
all_frames=[]
sort_by='firstSeen'
merged = pd.DataFrame()

for d in [processed_attack, processed_benign]:
        for f in os.listdir(d):
                if f.endswith('.csv'):
                        all_frames.append(pd.read_csv(os.path.join(d,f)))
if not all_frames:
        print('No CSV files found to merge.')
else:
        merged = pd.concat(all_frames, ignore_index=True)
        merged = merged.sort_values(by=sort_by)
        merged.to_csv(output, index=False)

# Statistiques

In [26]:
if not merged.empty:
        print(f"Total frames: {len(merged)}")
        print(f"Malicious frames: {len(merged[merged['label'] == 'MALICIOUS'])}")
        print(f"Benign frames: {len(merged[merged['label'] == 'BENIGN'])}")
else:
        print("Merged DataFrame is empty. No statistics to display.")

Total frames: 7285
Malicious frames: 2912
Benign frames: 4373
